# Prerequisites

In [1]:
# get data for labs
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8

File ‘around_the_world_in_80_days.txt’ already there; not retrieving.


# 1. Word Count

Instructions:  
For each cell marked "double-click and add explanation here" please answer the question in your own words.  
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.  
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [2]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
# Defind the rdd
rdd = sc.textFile('around_the_world_in_80_days.txt')

In [4]:
# view the first x lines of the rdd
rdd.take(5)

['The Project Gutenberg eBook of Around the World in Eighty Days',
 '    ',
 'This eBook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms']

In [5]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [6]:
# Note and explain the output of the below command
words

PythonRDD[3] at RDD at PythonRDD.scala:59

The output is not a list of words but a PythonRDD object. This is because flatMap is a transformation, and Spark works in a lazy way: it just records the operation to do, without running it right away. Nothing is actually computed until we call an action.

<ADD EXPLANATION HERE>

In [7]:
# Note and explain the output of the following command, focusing on the difference with the
# above command
words.take(20)

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for']

collect() is an action: it triggers the computation of the whole pipeline and brings all the elements of the RDD back to the driver as a Python list. That's the difference with the previous cell — words alone only showed the object (nothing was computed), while collect() runs it and returns the data.

In [8]:
# nicer print
for w in words.take(5):
    print(w)

The
Project
Gutenberg
eBook
of


In [9]:
# Print first x words
words.take(20)

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for']

In [10]:
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.

# flatMap applies the function to each line and then flattens the results into a single stream of words. With map we would get a list of lists (one list per line); with flatMap we get one flat sequence of words, which is what we need to count words one by one.

In [11]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

for w in words.take(20):
    print(w)

('The', 1)
('Project', 1)
('Gutenberg', 1)
('eBook', 1)
('of', 1)
('Around', 1)
('the', 1)
('World', 1)
('in', 1)
('Eighty', 1)
('Days', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('This', 1)
('eBook', 1)
('is', 1)
('for', 1)


In [12]:
# a. count the occurence of each word
words = rdd.flatMap(lambda lines: lines.split(' ')) \
           .map(lambda word: (word, 1))
word_counts = words.reduceByKey(lambda a, b: a + b)
word_counts.take(5)

[('Gutenberg', 60), ('eBook', 6), ('of', 1875), ('Around', 4), ('', 2193)]

In [13]:
# b. a common first step in text analysis, change all capital letters to lower case
words_lower = words.map(lambda x: (x[0].lower(), x[1]))
words_lower.take(5)

[('the', 1), ('project', 1), ('gutenberg', 1), ('ebook', 1), ('of', 1)]

In [14]:
# c. eliminate the stop words.
stop_words = ["a","an","and","are","as","at","be","by","for","from","has",
              "he","in","is","it","its","of","on","that","the","to","was",
              "were","will","with","this","you","your","i"]
words_no_stop = words_lower.filter(lambda x: x[0] not in stop_words)
words_no_stop.take(5)

[('project', 1), ('gutenberg', 1), ('ebook', 1), ('around', 1), ('world', 1)]

In [15]:
# d. sort in alphabetical order
words_sorted = words_no_stop.sortByKey()
words_sorted.take(5)

[('', 1), ('', 1), ('', 1), ('', 1), ('', 1)]

In [16]:
# e. sort descending by word frequency
words_counted = words_no_stop.reduceByKey(lambda a, b: a + b)
words_sorted_freq = words_counted.sortBy(lambda x: x[1], ascending=False)
words_sorted_freq.take(5)

[('', 2193), ('his', 855), ('not', 514), ('had', 503), ('which', 490)]

In [17]:
# f. remove punctuations and blank spaces
import string
words_clean = words_no_stop.map(
    lambda x: (x[0].translate(str.maketrans('', '', string.punctuation)).strip(),
               x[1])
).filter(lambda x: x[0] != "")
words_clean.take(5)

[('project', 1), ('gutenberg', 1), ('ebook', 1), ('around', 1), ('world', 1)]

In [19]:
import string

stop_words = ["a","an","and","are","as","at","be","by","for","from","has",
              "he","in","is","it","its","of","on","that","the","to","was",
              "were","will","with","this","you","your","i"]
punct_table = str.maketrans('', '', string.punctuation)

def word_count(rdd):
    return (rdd
        .flatMap(lambda line: line.split(' '))              # split into words
        .map(lambda w: w.translate(punct_table).strip())    # f: remove punctuation
        .map(lambda w: w.lower())                           # b: lower case
        .filter(lambda w: w != "" and w not in stop_words)  # c: remove stop words
        .map(lambda w: (w, 1))                              # (word, 1)
        .reduceByKey(lambda a, b: a + b))                   # a: count

word_counts = word_count(rdd)
word_counts.sortBy(lambda x: x[1], ascending=False).take(20)

[('his', 857),
 ('fogg', 577),
 ('not', 535),
 ('which', 515),
 ('had', 513),
 ('passepartout', 392),
 ('mr', 373),
 ('but', 326),
 ('him', 314),
 ('would', 278),
 ('have', 270),
 ('phileas', 250),
 ('they', 232),
 ('fix', 228),
 ('who', 199),
 ('or', 196),
 ('said', 194),
 ('her', 177),
 ('if', 172),
 ('been', 169)]

# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [ ]:
 # Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
  # (name, age) -> (name, (age, 1)) : we add a counter of 1 per person
  .map(lambda x: (x[0], (x[1], 1)))
  # group by name: sum the ages together, and sum the counters together
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # (name, (sum_ages, count)) -> (name, average) = sum_ages / count
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

## 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible


In [20]:
# Code here
import time

def timer(rdd):
    start = time.time()
    rdd.count()
    end = time.time()
    return end - start
timer(words_clean)

0.48851537704467773

In [22]:
#The best order is to clean and filter first (remove punctuation, lower case, remove stop words), 
#then count with reduceByKey, and sort last.
#reduceByKey and sortBy trigger a shuffle (data is moved between partitions), which is the most expensive 
#operation in Spark. So we want to reduce the amount of data before the shuffle. Cleaning and filtering 
#happen with map and filter, which do not need a shuffle, so counting then works on far fewer words. 
#On this small file the two times are very close, but on a large dataset Order A (clean then count) 
#scales better because it shuffles less data.


## 4. Text Comparison

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two

In [23]:
# Code here
!wget -nc -O around_the_world_in_80_days_fr.txt https://www.gutenberg.org/ebooks/46541.txt.utf-8

--2026-09-17 15:07:33--  https://www.gutenberg.org/ebooks/46541.txt.utf-8
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 152.19.134.47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: http://www.gutenberg.org/cache/epub/46541/pg46541.txt [following]
URL transformed to HTTPS due to an HSTS policy
--2026-09-17 15:07:33--  https://www.gutenberg.org/cache/epub/46541/pg46541.txt
Reusing existing connection to www.gutenberg.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 472731 (462K) [text/plain]
Saving to: ‘around_the_world_in_80_days_fr.txt’

around_the_world_in 100%[===================>] 461.65K   390KB/s    in 1.2s    

2026-09-17 15:07:35 (390 KB/s) - ‘around_the_world_in_80_days_fr.txt’ saved [472731/472731]



In [24]:
rdd_french = sc.textFile('around_the_world_in_80_days_fr.txt')
rdd_french.take(5)

['The Project Gutenberg eBook of Le Tour du monde en quatre-vingts jours',
 '    ',
 'This eBook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms']

In [25]:
stop_words_fr = ["le","la","les","un","une","des","de","du","et","ou","a",
                 "au","aux","en","dans","sur","pour","par","avec","sans",
                 "ce","cet","cette","ces","qui","que","quoi","dont","il",
                 "elle","ils","elles","je","tu","nous","vous","se","son",
                 "sa","ses","leur","leurs","est","sont","ont","ne","pas","plus"]

In [26]:
def word_count_fr(rdd):
    return (rdd
        .flatMap(lambda line: line.split(' '))
        .map(lambda w: w.translate(punct_table).strip())
        .map(lambda w: w.lower())
        .filter(lambda w: w != "" and w not in stop_words_fr)
        .map(lambda w: (w, 1))
        .reduceByKey(lambda a, b: a + b))

word_counts_french = word_count_fr(rdd_french)
word_counts_french.sortBy(lambda x: x[1], ascending=False).take(20)

[('à', 1597),
 ('fogg', 681),
 ('passepartout', 449),
 ('mais', 360),
 ('lui', 333),
 ('phileas', 328),
 ('était', 320),
 ('mr', 287),
 ('fix', 284),
 ('avait', 280),
 ('heures', 242),
 ('on', 234),
 ('répondit', 215),
 ('quil', 194),
 ('tout', 192),
 ('the', 188),
 ('bien', 181),
 ('si', 181),
 ('comme', 170),
 ('deux', 153)]

In [27]:
# Compare the two versions
# word_counts = English counts (from the final function in part 1)

total_fr = word_counts_french.map(lambda x: x[1]).sum()
total_en = word_counts.map(lambda x: x[1]).sum()
print("Total words FR:", total_fr)
print("Total words EN:", total_en)

unique_fr = word_counts_french.count()
unique_en = word_counts.count()
print("Distinct words FR:", unique_fr)
print("Distinct words EN:", unique_en)

print("--- Top 10 FR ---")
for w in word_counts_french.sortBy(lambda x: x[1], ascending=False).take(10):
    print(w)
print("--- Top 10 EN ---")
for w in word_counts.sortBy(lambda x: x[1], ascending=False).take(10):
    print(w)

Total words FR: 48381
Total words EN: 44524
Distinct words FR: 11153
Distinct words EN: 8281
--- Top 10 FR ---
('à', 1597)
('fogg', 681)
('passepartout', 449)
('mais', 360)
('lui', 333)
('phileas', 328)
('était', 320)
('mr', 287)
('fix', 284)
('avait', 280)
--- Top 10 EN ---
('his', 857)
('fogg', 577)
('not', 535)
('which', 515)
('had', 513)
('passepartout', 392)
('mr', 373)
('but', 326)
('him', 314)
('would', 278)
